In [ ]:
#Download Libs
import numpy as np
from torchvision.datasets import CIFAR10
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


In [ ]:

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here: Read the dataset Q3_data.csv using read_csv()
gf_path = os.path.join(path, 'Q3_data.csv')
df_gf = pd.read_csv(gf_path)

print(f"Shape: {df_gf.shape}")

In [ ]:
# Task 2: Write your code here: Inspect the first few rows using head()
df_gf.head()

In [ ]:
# Task 3: Write your code here: Display dataset information using info()
df_gf.info()

In [ ]:
# Task 4: Write your code here: Show statistical description using describe()
df_gf.describe()

In [ ]:
# Task 1: Write your code here: Handle missing values appropriately
def check_missing_values(df):

  # Get missing values using pandas
  missing_values = df.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_gf)

In [ ]:
# Task 2: Write your code here: Check and remove duplicates
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_gf)

In [ ]:
# Task 3: Write your code here: Encode categorical variables
categorical_cols = df_gf.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()

    df_gf[col] = le.fit_transform(df_gf[col])

df_gf.head()

In [ ]:
# Task 4: Write your code here:Apply feature scaling
numerical_cols =  df_gf.select_dtypes(include=["number"]).columns.drop("Delivery_Time")
scaler = StandardScaler()
df_gf[numerical_cols] = scaler.fit_transform(df_gf[numerical_cols])
df_gf.head()

In [ ]:
# Task 5: Write your code here: Check for target imbalance and state if it is imbalanced or not
#used method to check the imbalance
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df_gf, "golden feature")

In [ ]:
# Task 1: Write your code here:
X = df_gf.drop('golden feature', axis=1).astype(float)
y = df_gf['golden feature'].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from  catBoost import CatBoostClassifier
# Task 2,3,4,5: Write your code here:
#used K-fold and trained the model with  CatBoostClassifier
#2
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
#3
"CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
#4
# Storage for logistic regression results for each fold
lr_losses = []
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    theta, losses = gradient_descent(X_train, y_train, learning_rate=0.5, n_iters=500)

    # Validate
    y_pred_proba = sigmoid(np.dot(X_test, theta))
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    #5
    lr_losses.append(losses)
    lr_accuracy.append(accuracy)
    lr_precision.append(precision)
    lr_recall.append(recall)
    lr_f1.append(f1)

In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_gf['golden_feature'].dropna(), bins=30, edgecolor='black')
plt.title('golden feature Distribution')
plt.xlabel('golden feature')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: